In [3]:
!pip install pandas openpyxl xlrd

import pandas as pd

# ---------------- CONFIG ----------------
BRENT_FILE = "BrentSpotPrice.xls"
CRUDE_FILE = "USCrude_Stock.xls"
USD_FILE = "Dollar_Index.xlsx"
PMI_FILE = "ChinaPMI.xlsx"
GPR_FILE = "data_gpr_export.xls"

START = "2010-01-01"
END = "2026-07-01"
OUTPUT_FILE = "merged_monthly.csv"
# -----------------------------------------



[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
def load_brent(path):
    """EIA Brent file: 'Data 1' sheet, 3 header rows, then Date | Price."""
    raw = pd.read_excel(path, sheet_name="Data 1", header=None)
    df = raw.iloc[3:, :2].copy()
    df.columns = ["Date", "Brent_USD_bbl"]
    df["Date"] = pd.to_datetime(df["Date"])
    df["Brent_USD_bbl"] = pd.to_numeric(df["Brent_USD_bbl"], errors="coerce")
    df = df.set_index("Date").resample("MS").mean()  # already monthly
    return df.loc[START:END]

In [5]:
def load_crude_stocks(path):
    """EIA weekly stocks file: 26 series, resampled to monthly average."""
    raw = pd.read_excel(path, sheet_name="Data 1", header=None)
    codes = raw.iloc[1].tolist()          # EIA series codes (row 1)
    data = raw.iloc[3:].copy()            # actual weekly data starts row 3
    data.columns = codes
    data["Date"] = pd.to_datetime(data["Sourcekey"])
    data = data.set_index("Date")

    # Pick the series
    keep = {
        "WCRSTUS1": "US_Crude_Stocks_Total_kbbl"
    }
    df = data[list(keep.keys())].apply(pd.to_numeric, errors="coerce")
    df = df.rename(columns=keep)

   # "last" = end-of-month inventory level, i.e. the last weekly report
    # falling within each calendar month (not an average of the month's readings)
    df = df.resample("ME").last()
    df.index = df.index.to_period("M").to_timestamp()  # align to month-start, like other series
    return df.loc[START:END]

In [6]:
def load_usd_index(path):
    """FRED DTWEXBGS file: 'Monthly' sheet, already clean."""
    df = pd.read_excel(path, sheet_name="Monthly")
    df["observation_date"] = pd.to_datetime(df["observation_date"])
    df = df.rename(columns={"observation_date": "Date", "DTWEXBGS": "USD_Index"})
    df = df.set_index("Date").resample("MS").mean()
    return df.loc[START:END]

In [7]:
def load_china_pmi(path):
    
    raw = pd.read_excel(path, sheet_name="Sheet1", header=None)
    date_row = raw.iloc[2, 1:]                       # e.g. "Jul 2026", "Jun 2026", ...
    indicators = raw.iloc[3:17, 0].str.strip().tolist()
    values = raw.iloc[3:17, 1:]
    values.index = indicators
    values.columns = date_row.values

    df = values.T
    df.index = pd.to_datetime(df.index, format="%b %Y")
    df = df[~df.index.duplicated(keep="first")].sort_index()
    df = df.apply(pd.to_numeric, errors="coerce")

    keep = {
        "Manufacturing Purchasing Managers' Index (%)": "China_PMI_Manufacturing",
    }
    df = df[list(keep.keys())].rename(columns={k.strip(): v for k, v in keep.items()})
    df.index.name = "Date"
    return df.loc[START:END]

In [8]:
def load_gpr(path):
    """Caldara & Iacoviello GPR export: keep only the headline Recent GPR index."""
    raw = pd.read_excel(path, sheet_name="Sheet1")
    df = raw[["month", "GPR"]].copy()
    df["month"] = pd.to_datetime(df["month"])
    df = df.rename(columns={"month": "Date", "GPR": "GPR_Index"})
    df = df.set_index("Date")
    return df.loc[START:END]

In [10]:
import os
print("Current directory:", os.getcwd())
print("Files here:", os.listdir())

Current directory: C:\Users\jiech\PycharmProjects\JupyterProject
Files here: ['.idea', '.venv', 'carbon-externality.ipynb', 'cpi_retail.ipynb', 'cpi_retail_clean.ipynb', 'cpi_retail_wage_popu.ipynb', 'Crude oil.ipynb', 'data', 'inflation-retail-github.ipynb', 'models', 'pandas&numpy.ipynb', 'pigouvian_tax_graph.png', 'README.md', 'requirements.txt', 'timeseries.ipynb', 'usd-to-sgd.ipynb']


In [14]:
def main():
    brent = load_brent(BRENT_FILE)
    crude = load_crude_stocks(CRUDE_FILE)
    usd = load_usd_index(USD_FILE)
    pmi = load_china_pmi(PMI_FILE)
    gpr = load_gpr(GPR_FILE)

    merged = brent.join([crude, usd, pmi, gpr], how="outer")
    merged = merged.loc[START:END].reset_index().rename(columns={"index": "Date"})

    print("Merged shape:", merged.shape)
    print("\nMissing values per column:")
    print(merged.isna().sum())

    merged.to_csv(OUTPUT_FILE, index=False)
    print(f"\nSaved -> {OUTPUT_FILE}")


if __name__ == "__main__":
    main()

Merged shape: (199, 6)

Missing values per column:
Date                          0
Brent_USD_bbl                 0
US_Crude_Stocks_Total_kbbl    0
USD_Index                     0
China_PMI_Manufacturing       0
GPR_Index                     0
dtype: int64

Saved -> merged_monthly.csv
